## **Problem:**
Use the `credit.csv` dataset to build **classification model** using KNN,NB. The target variable is `default` which is a binary label to indicate of the loan is default (yes, no). Use all other variables for your feature set.


- Comment your code, do not display unnecessary data and keep your notebook clean and readable.

### **Read `credit.csv` into a dataframe `credit_df`. Display the first 5 rows**

- Handel the missing value character '?' as na value.*
- Remove whitespace from column names and replace with underscore

In [1]:
import pandas as pd
import numpy as np

credit_df = pd.read_csv('credit_loan.csv', na_values='?')

In [2]:
# Clean column names.
credit_df.columns = credit_df.columns.str.strip()
credit_df.columns = credit_df.columns.str.replace(r'\s+', '_', regex=True)

In [3]:
credit_df.head()

,months_loan_duration,credit_history,amount,percent_of_income,years_at_residence,age,existing_loans_count,job,dependents,phone,default
0,6,critical,1169,4.0,4.0,67.0,2,skilled,1,yes,no
1,48,good,5951,2.0,2.0,22.0,1,skilled,1,no,yes
2,12,critical,2096,2.0,3.0,49.0,1,unskilled,2,no,no
3,42,good,7882,2.0,4.0,45.0,1,skilled,2,no,no
4,24,poor,4870,NaN,4.0,53.0,2,skilled,2,no,yes


### **1. Display the following in each cell**
- The balance of your target variable (no vs yes count)
- Missing values per column,
- Total number of rows that have missing values

In [4]:
# Count each target class.
credit_df['default'].value_counts()

,count
default,
no,700
yes,300


In [5]:
# Count missing values in each column.
credit_df.isna().sum()

,0
months_loan_duration,0
credit_history,3
amount,0
percent_of_income,4
years_at_residence,6
age,1
existing_loans_count,0
job,0
dependents,0
phone,0


In [6]:
# Count rows that contain missing values.
credit_df.isna().any(axis=1).sum()

np.int64(12)

### **2. Apply the proper preprocessing steps to the dataframe.`**


In [7]:
# Numeric columns
num_cols = ['months_loan_duration', 'amount', 'percent_of_income',
            'years_at_residence', 'age', 'existing_loans_count',
            'dependents']

# Categorical columns
cat_cols = ['credit_history', 'job', 'phone']

In [8]:
# Convert numeric columns
credit_df[num_cols] = credit_df[num_cols].apply(pd.to_numeric, errors='coerce')

# Fill missing numeric values with mean
credit_df[num_cols] = credit_df[num_cols].fillna(credit_df[num_cols].mean())

# Fill missing categorical values with mode
credit_df[cat_cols] = credit_df[cat_cols].fillna(credit_df[cat_cols].mode().iloc[0])

In [9]:
credit_df.head()

,months_loan_duration,credit_history,amount,percent_of_income,years_at_residence,age,existing_loans_count,job,dependents,phone,default
0,6,critical,1169,4.000000,4.0,67.0,2,skilled,1,yes,no
1,48,good,5951,2.000000,2.0,22.0,1,skilled,1,no,yes
2,12,critical,2096,2.000000,3.0,49.0,1,unskilled,2,no,no
3,42,good,7882,2.000000,4.0,45.0,1,skilled,2,no,no
4,24,poor,4870,2.971888,4.0,53.0,2,skilled,2,no,yes


In [10]:
credit_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   months_loan_duration  1000 non-null   int64  
 1   credit_history        1000 non-null   object 
 2   amount                1000 non-null   int64  
 3   percent_of_income     1000 non-null   float64
 4   years_at_residence    1000 non-null   float64
 5   age                   1000 non-null   float64
 6   existing_loans_count  1000 non-null   int64  
 7   job                   1000 non-null   object 
 8   dependents            1000 non-null   int64  
 9   phone                 1000 non-null   object 
 10  default               1000 non-null   object 
dtypes: float64(3), int64(4), object(4)
memory usage: 86.1+ KB


In [11]:
# Convert categorical variables into dummy variables
credit_df = pd.get_dummies(credit_df, columns=cat_cols, drop_first=True, dtype=int)

In [12]:
credit_df.head()

,months_loan_duration,amount,percent_of_income,years_at_residence,age,existing_loans_count,dependents,default,credit_history_good,credit_history_perfect,credit_history_poor,credit_history_very good,job_skilled,job_unemployed,job_unskilled,phone_yes
0,6,1169,4.000000,4.0,67.0,2,1,no,0,0,0,0,1,0,0,1
1,48,5951,2.000000,2.0,22.0,1,1,yes,1,0,0,0,1,0,0,0
2,12,2096,2.000000,3.0,49.0,1,2,no,0,0,0,0,0,0,1,0
3,42,7882,2.000000,4.0,45.0,1,2,no,1,0,0,0,1,0,0,0
4,24,4870,2.971888,4.0,53.0,2,2,yes,0,0,1,0,1,0,0,0


### **3. Partition the dataset into `X` feature matrix and `y` target variable.**

In [13]:
# Separate features and target.
X = credit_df.drop(columns='default')
y = credit_df['default']

### **4. Partition the data into training and testing data set and apply MinMax Scaling**
- Use train_test_split with 30% for testing size and apply random_state = 23

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=23)

In [15]:
# Create the scaler.
scaler = MinMaxScaler()

In [16]:
# Fit on training data and transform both sets.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### **5. Build a k-nearest neighbors (KNN) classifier and train the model using the using GridSearchCV**

In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

In [18]:
# Create a KNN model.
knn = KNeighborsClassifier()

In [19]:
# Choose values for K.
knn_parameters = {'n_neighbors': [3, 5, 7, 9]}

In [20]:
# Use grid search to find the best K.
knn_grid = GridSearchCV(knn, param_grid=knn_parameters, cv=5)

In [21]:
# Train KNN.
knn_grid.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 5, 7, 9]})

In [22]:
# Show the best K.
knn_grid.best_params_

{'n_neighbors': 5}

In [23]:
# Predict with KNN.
knn_pred = knn_grid.predict(X_test_scaled)

### **6. Build a MultinomialNB (MNB) classifier and train the model using the using GridSearchCV**

In [24]:
from sklearn.naive_bayes import MultinomialNB

In [25]:
# Create an MNB model.
mnb = MultinomialNB()

In [26]:
# Choose alpha values.
mnb_parameters = {'alpha': [0.1, 1, 10]}

In [27]:
# Use grid search to find the best alpha.
mnb_grid = GridSearchCV(mnb, param_grid=mnb_parameters, cv=5)

In [28]:
# Train MNB.
mnb_grid.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=MultinomialNB(),
             param_grid={'alpha': [0.1, 1, 10]})

In [29]:
# Show the best alpha.
mnb_grid.best_params_

{'alpha': 1}

In [30]:
# Predict with MNB.
mnb_pred = mnb_grid.predict(X_test_scaled)

### **7. Evaluate performance of both models and discuss the results indicating which model provides better performance**

In [31]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score

#### KNN Classfication Report

In [32]:
# Show the KNN classification report.
print(classification_report(y_test, knn_pred, zero_division=0))

              precision    recall  f1-score   support

          no       0.70      0.86      0.77       206
         yes       0.39      0.19      0.26        94

    accuracy                           0.65       300
   macro avg       0.55      0.53      0.52       300
weighted avg       0.60      0.65      0.61       300



#### MNB CLassification Report

In [33]:
# Show the MNB classification report.
print(classification_report(y_test, mnb_pred, zero_division=0))

              precision    recall  f1-score   support

          no       0.69      0.97      0.81       206
         yes       0.40      0.04      0.08        94

    accuracy                           0.68       300
   macro avg       0.54      0.51      0.44       300
weighted avg       0.60      0.68      0.58       300



#### Summarize the Performance results:



In [34]:
# Compare test accuracy.
knn_accuracy = accuracy_score(y_test, knn_pred)
mnb_accuracy = accuracy_score(y_test, mnb_pred)
print('KNN accuracy:', round(knn_accuracy, 3))
print('MNB accuracy:', round(mnb_accuracy, 3))

KNN accuracy: 0.653
MNB accuracy: 0.68


In [35]:
# Compare macro F1-scores.
knn_f1 = f1_score(y_test, knn_pred, average='macro', zero_division=0)
mnb_f1 = f1_score(y_test, mnb_pred, average='macro', zero_division=0)
print('KNN macro F1:', round(knn_f1, 3))
print('MNB macro F1:', round(mnb_f1, 3))

KNN macro F1: 0.516
MNB macro F1: 0.442


MNB has a higher accuracy (0.680) than KNN (0.653). However, KNN has a higher macro F1-score (0.516 vs. 0.442) and performs better at predicting the "yes" class. Overall, KNN is better at balancing performance between the two classes.
